In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial import distance
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [ ]:
root = "/home/acomajuncosa/Documents_GPU/mtb-targeted-protein-degradation/notebooks"

perc = "1"

# Defining path to Enamine screening results and path to output
PATH_TO_RESULTS = "/aloy/home/acomajuncosa/Ersilia/gcadda4tb-enamine-real-screening/results"
PATH_TO_OUTPUT = os.path.join(root, "..", "processed", 'unidock_REAL_docking', 'inference_10B')
os.makedirs(os.path.join(PATH_TO_OUTPUT, "shared_compounds"), exist_ok=True)

# Load chunk info
CHUNKS = pd.read_csv("/aloy/home/acomajuncosa/Ersilia/gcadda4tb-enamine-real-screening/data/chunks/chunks.csv", header=None)[0].tolist()

# Load pocket info
POCKETS = sorted(set(sorted([i.split("_ind_")[0] for i in sorted(os.listdir(os.path.join(PATH_TO_RESULTS, "Enamine_REAL_LeadLike_000")))])))

In [3]:
# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))
pocket_to_coords = {}

for file, pocket_numb, coord in zip(pocket_detection_data['File name'], pocket_detection_data['Pocket number'], pocket_detection_data['Pocket centroid coordinate (x y z)']):
    
    # Prepare labels
    label = file.replace(".pdb", "") + "_pocket_" + str(pocket_numb)

    # Create mappings
    pocket_to_coords[label] = np.array(coord.split(), dtype=float)

# Identify pairs
diff_protein, same_protein, same_pocket = set(), set(), set()

for c, pocket1 in tqdm(enumerate(POCKETS)):
    for pocket2 in POCKETS[c+1:]:

        # Define pair
        pair = (pocket1, pocket2)
        
        # If the protein is different
        if pocket1.split("_")[1] != pocket2.split("_")[1]:
                diff_protein.add(pair)

        else:

            # Calculate distance
            d = distance.euclidean(pocket_to_coords[pocket1], pocket_to_coords[pocket2])

            # If the pocket is the same
            if d < 6.14:
                same_pocket.add(pair)

            # If the pocket is not the same
            else:
                same_protein.add(pair)

print(f"Same pocket: {len(same_pocket)}")
print(f"Same protein: {len(same_protein)}")
print(f"Different protein: {len(diff_protein)}")

276it [00:00, 9659.05it/s]

Same pocket: 738
Same protein: 1639
Different protein: 35573


In [11]:
# For each chunk
for chunk in CHUNKS[1:]:

    TOPs = {}
    SHARED_HITS = []

    # For each pocket
    for pocket in tqdm(POCKETS):

        # Get indices at specified percentile
        inds = np.load(os.path.join(PATH_TO_RESULTS, chunk, f"{pocket}_ind_{perc}.npz"))[f"ind_{perc}"]
        cut = float(np.load(os.path.join(PATH_TO_RESULTS, chunk, f"{pocket}_ind_{perc}.npz"))["thr"])

        # Store indices
        TOPs[pocket] = set(inds.tolist())

    # Evaluate shared hits among pairs
    SHARED_HITS_SAME_POCKET = []
    SHARED_HITS_SAME_PROTEIN = []
    SHARED_HITS_DIFF_PROTEIN = []

    # Calculate intersections
    for c1, pocket1 in tqdm(enumerate(sorted(TOPs))):
        for pocket2 in sorted(TOPs)[c1+1:]:
            intersection = len(TOPs[pocket1].intersection(TOPs[pocket2]))
            if (pocket1, pocket2) in same_pocket:
                SHARED_HITS_SAME_POCKET.append(intersection)
            if (pocket1, pocket2) in same_protein:
                SHARED_HITS_SAME_PROTEIN.append(intersection)
            if (pocket1, pocket2) in diff_protein:
                SHARED_HITS_DIFF_PROTEIN.append(intersection)

    # Save chunk results
    SHARED_HITS.append([chunk, perc, "SAME_POCKET", len(SHARED_HITS_SAME_POCKET), ";".join([str(k) for k in sorted(SHARED_HITS_SAME_POCKET)])])
    SHARED_HITS.append([chunk, perc, "SAME_PROTEIN", len(SHARED_HITS_SAME_PROTEIN), ";".join([str(k) for k in sorted(SHARED_HITS_SAME_PROTEIN)])])
    SHARED_HITS.append([chunk, perc, "DIFF_PROTEIN", len(SHARED_HITS_DIFF_PROTEIN), ";".join([str(k) for k in sorted(SHARED_HITS_DIFF_PROTEIN)])])
    SHARED_HITS = pd.DataFrame(SHARED_HITS, columns=['chunk', 'perc', 'set', 'count', 'indices'])
    SHARED_HITS.to_csv(os.path.join(PATH_TO_OUTPUT, "shared_compounds", f"{chunk}.csv"), index=False)

100%|██████████| 276/276 [01:48<00:00,  2.53it/s]
276it [02:17,  2.01it/s]
100%|██████████| 276/276 [01:44<00:00,  2.64it/s]
276it [02:16,  2.02it/s]
100%|██████████| 276/276 [01:40<00:00,  2.75it/s]
276it [02:18,  1.99it/s]
100%|██████████| 276/276 [01:42<00:00,  2.70it/s]
276it [02:14,  2.04it/s]
100%|██████████| 276/276 [01:39<00:00,  2.77it/s]
276it [02:17,  2.01it/s]
100%|██████████| 276/276 [01:39<00:00,  2.78it/s]
276it [02:17,  2.00it/s]
100%|██████████| 276/276 [01:47<00:00,  2.57it/s]
276it [02:16,  2.03it/s]
100%|██████████| 276/276 [01:49<00:00,  2.53it/s]
276it [02:19,  1.98it/s]
100%|██████████| 276/276 [01:45<00:00,  2.61it/s]
276it [02:16,  2.03it/s]
100%|██████████| 276/276 [01:43<00:00,  2.67it/s]
276it [02:14,  2.05it/s]
100%|██████████| 276/276 [01:46<00:00,  2.58it/s]
276it [02:16,  2.02it/s]
100%|██████████| 276/276 [01:50<00:00,  2.49it/s]
276it [02:17,  2.01it/s]
100%|██████████| 276/276 [01:43<00:00,  2.66it/s]
276it [02:18,  2.00it/s]
100%|██████████| 276/276 

KeyboardInterrupt: 

In [ ]:
bins = [i for i in range(0, 100_000, 2_500)]
h = SHARED_HITS_SAME_POCKET + SHARED_HITS_SAME_PROTEIN + SHARED_HITS_DIFF_PROTEIN

plt.hist(h, bins=bins, edgecolor='k', linewidth=1, zorder=2, density=False)
plt.grid(linestyle='--', zorder=-2)
plt.ylabel("Pocket pairs", labelpad=12)
plt.xlabel("Number of shared hits", labelpad=12)
plt.show()

In [ ]:
bins = [i for i in range(0, 100_000, 2_500)]

def fmt_k(x):
    x = float(x)
    return f"{x/1000:.1f}k" if abs(x) >= 1000 else f"{x:.1f}"

label1 = f"Diff protein ({len(SHARED_HITS_DIFF_PROTEIN)}) / Mean: {fmt_k(np.mean(SHARED_HITS_DIFF_PROTEIN))} / Med: {fmt_k(np.median(SHARED_HITS_DIFF_PROTEIN))}"
label2 = f"Same prot - diff pocket ({len(SHARED_HITS_SAME_PROTEIN)}) / Mean: {fmt_k(np.mean(SHARED_HITS_SAME_PROTEIN))} / Med: {fmt_k(np.median(SHARED_HITS_SAME_PROTEIN))}"
label3 = f"Same pocket ({len(SHARED_HITS_SAME_POCKET)}) / Mean: {fmt_k(np.mean(SHARED_HITS_SAME_POCKET))} / Med: {fmt_k(np.median(SHARED_HITS_SAME_POCKET))}"

plt.hist(SHARED_HITS_DIFF_PROTEIN, bins=bins, edgecolor='k', linewidth=1, color="#50285A", label=label1, zorder=2, density=True)
plt.hist(SHARED_HITS_SAME_PROTEIN, bins=bins, edgecolor='k', linewidth=1, color="#FAD782", label=label2, zorder=2, density=True, alpha=0.8)
plt.hist(SHARED_HITS_SAME_POCKET, bins=bins, edgecolor='k', linewidth=1, color="#BEE6B4", label=label3, zorder=2, density=True, alpha=0.7)

l = plt.legend(framealpha=1, edgecolor='k', prop={'size': 10})
l.get_frame().set_linewidth(0.6)
plt.grid(linestyle='--', zorder=-2)
plt.ylabel("Pocket pairs", labelpad=12)
plt.xlabel("Number of shared hits", labelpad=12)
plt.show()

In [ ]:
from collections import Counter
counting_cpds = Counter([ind for pocket in sorted(TOPs) for ind in TOPs[pocket]])

In [ ]:
len([i for i in counting_cpds if counting_cpds[i] > 0])

In [ ]:
len([i for i in counting_cpds if counting_cpds[i] > 200])